In [ ]:
import pandas as pd
import plotly.express as px

# Carica i file CSV in DataFrame Pandas
movies = pd.read_csv('data/movies.csv')

# Pulisco i dati
movies.drop_duplicates(inplace=True)
movies.dropna(inplace=True)
movies.fillna(0, inplace=True)
movies.to_csv('data/clean_movies.csv', index=False)

# Leggi il file movies.csv
movies = pd.read_csv('data/clean_movies.csv')

countries = pd.read_csv('data/countries.csv')

# Pulisco i dati
countries.drop_duplicates(inplace=True)
countries.dropna(inplace=True)
countries.fillna(0, inplace=True)
countries.to_csv('data/clean_countries.csv', index=False)

countries = pd.read_csv('data/clean_countries.csv')


"""
Carico i due file CSV, unisco i due DataFrame sulla colonna id, che rappresenta l'ID del film, per ottenere un DataFrame combinato con informazioni sia sui film che sui paesi. Raggruppo i dati per paese e conto il numero di film per ogni paese, salvando il risultato in un nuovo DataFrame chiamato movie_counts_by_country
"""
# Unisci i DataFrame movies e countries sulla colonna 'id'
movies_countries = pd.merge(movies, countries, on='id')

# Raggruppa per paese e conta il numero di film
movie_counts_by_country = movies_countries['country'].value_counts().reset_index()
movie_counts_by_country.columns = ['country', 'movie_count']

"""
Crea una mappa coropletica usando il DataFrame movie_counts_by_country.
 - locations='country' specifica che la colonna country contiene i nomi dei paesi
 - locationmode='country names' indica che i nomi dei paesi sono usati per posizionarli sulla mappa
 - color='movie_count' usa la colonna movie_count per colorare i paesi in base al numero di film
 - color_continuous_scale='Viridis' sceglie una scala di colori continua per la mappa
 prova

"""

# Crea la mappa coropletica
fig = px.choropleth(movie_counts_by_country,
                    locations='country',
                    locationmode='country names',
                    color='movie_count',
                    color_continuous_scale='Viridis',
                    title='Distribuzione geografica dei film')

# Mostro la mappa
fig.show()

releases = pd.read_csv('data/releases.csv')

# Pulisco i dati
releases.drop_duplicates(inplace=True)
releases.dropna(inplace=True)
releases.fillna(0, inplace=True)
releases.to_csv('data/clean_releases.csv', index=False)

releases = pd.read_csv('data/clean_releases.csv')

# Unisco i DataFrame movies e releases sulla colonna 'id'
movies_releases = pd.merge(movies, releases, on='id')

"""
Una volta uniti i Dataframe avrò date_x cje rappresenta solo l'anno, per i movies
e date_y nel formato YYYY_MM_DD del dataset releases
"""

"""
# Visualizza le prime 5 righe
print(movies_releases.head().to_markdown(index=False, numalign="left", stralign="left"))

# Visualizza le colonne e i loro tipi
print(movies_releases.info())
"""

movies_releases['date_y'] = pd.to_datetime(movies_releases['date_y'], format='%Y-%m-%d', errors='coerce')


# Raggruppa per paese e calcola la media del rating (usando la colonna 'rating' da movies)
average_rating_by_country = movies_releases.groupby('country')['rating_x'].mean().reset_index()

# Crea la mappa coropletica
fig = px.choropleth(average_rating_by_country,
                    locations='country',
                    locationmode='country names',
                    color='rating_x',  # Usa 'rating_x' invece di 'rating'
                    color_continuous_scale='Viridis',
                    title='Rating medio dei film per paese')

# Mostra la mappa
fig.show()

# Estraggo l'anno di uscita dalla colonna 'date'
movies_releases['release_year'] = movies_releases['date_y'].dt.year

# Converto l'anno in un oggetto datetime con formato YYYY, perchè la 'date' di movies_releases è  nel formato YYYY-MM-DD
movies_releases['release_year_dt'] = pd.to_datetime(movies_releases['release_year'], format='%Y')

"""
# Calcola la differenza in giorni tra la data di uscita e la data di uscita originale
movies_releases['days_since_original_release'] = (movies_releases['date_y'] - movies['date']).dt.days

movies_releases['days_since_original_release'] = (movies_releases['release_year_dt'] - pd.to_datetime(movies['date'], format='%Y')).dt.days
"""

movies_releases['years_since_original_release'] = movies_releases['release_year'] - pd.to_datetime(movies['date'], format='%Y').dt.year

# Raggruppa per paese e anno di uscita, e calcola la media del rating e la media della differenza in anni
#average_rating_by_country_year = movies_releases.groupby(['country', 'release_year'])[['rating_x', 'years_since_original_release']].mean().reset_index()

average_rating_by_country_year = movies_releases.groupby(['country', 'release_year'])[['rating_x', 'years_since_original_release', 'name']].agg({'rating_x': 'mean', 'years_since_original_release': 'mean', 'name': lambda x: list(x)}).reset_index()


# Crea un grafico a dispersione per visualizzare la correlazione
fig = px.scatter(average_rating_by_country_year,
                 x='years_since_original_release',
                 y='rating_x',
                 color='country',
                 title='Correlazione tra data di uscita e rating',
                 labels={'years_since_original_release': 'Anni dalla prima uscita',
                         'rating_x': 'Rating medio'},
                 hover_data={'name': True})  # Aggiungo il nome del film ai dati hover
 # Mostra il grafico
fig.show()